# Module 3: Spark RDDs

Run in Google Colab or a local PySpark environment.

In [ ]:
# Run this cell first
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("LearningModules").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Load datasets (adjust path if needed)
students_df = spark.read.csv("../data/students.csv", header=True, inferSchema=True)
enrollments_df = spark.read.csv("../data/enrollments.csv", header=True, inferSchema=True)

## Creating RDDs with parallelize()

RDDs (Resilient Distributed Datasets) are the fundamental data structure in Spark. You can create them from Python collections.

In [ ]:
# Create an RDD from student tuples
student_tuples = [
    (1, "Alice", 3.8),
    (2, "Bob", 3.2),
    (3, "Charlie", 3.9),
    (4, "Diana", 2.7),
    (5, "Eve", 3.5)
]

students_rdd = sc.parallelize(student_tuples)
print(f"RDD created with {students_rdd.count()} elements")
print(f"Number of partitions: {students_rdd.getNumPartitions()}")

## Transformations: map, filter, flatMap

Transformations create new RDDs from existing ones. They are lazy (not executed until an action is called).

In [ ]:
# map: extract names
names_rdd = students_rdd.map(lambda x: x[1])
print("Names:", names_rdd.collect())

In [ ]:
# filter: students with GPA > 3.5
high_gpa_rdd = students_rdd.filter(lambda x: x[2] > 3.5)
print("High GPA students:", high_gpa_rdd.collect())

In [ ]:
# flatMap: split names into characters
chars_rdd = names_rdd.flatMap(lambda name: list(name))
print("First 10 chars:", chars_rdd.take(10))

## Pair RDDs and reduceByKey

Pair RDDs contain key-value tuples and support aggregation operations like `reduceByKey`.

In [ ]:
# Create a pair RDD: (gpa_category, count)
gpa_categories = students_rdd.map(
    lambda x: ("High" if x[2] >= 3.5 else "Standard", 1)
)

# reduceByKey to count per category
category_counts = gpa_categories.reduceByKey(lambda a, b: a + b)
print("Category counts:", category_counts.collect())

## Actions: collect, count, take

Actions trigger computation and return results to the driver.

In [ ]:
# collect: returns all elements
all_students = students_rdd.collect()
print("All:", all_students)

# count: number of elements
print("Count:", students_rdd.count())

# take: return first n elements
print("First 2:", students_rdd.take(2))

# first: return the first element
print("First:", students_rdd.first())

## Practice Problem 1: Map and Filter

Using the students_rdd, create a new RDD containing only the names of students with GPA above 3.4.

<details><summary>Hint</summary>Chain filter() to keep GPAs above 3.4, then map() to extract names.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
result = students_rdd.filter(lambda x: x[2] > 3.4).map(lambda x: x[1])
print(result.collect())

## Practice Problem 2: reduceByKey

Create a pair RDD of (name_length, gpa) and use reduceByKey to find the total GPA for each name length group.

<details><summary>Hint</summary>Map each student to (len(name), gpa) then reduceByKey with addition.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
length_gpa = students_rdd.map(lambda x: (len(x[1]), x[2]))
totals = length_gpa.reduceByKey(lambda a, b: a + b)
print(totals.collect())

## Practice Problem 3: Word Count with flatMap

Given the following sentences RDD, count the occurrences of each word.

<details><summary>Hint</summary>Use flatMap to split sentences into words, map each word to (word, 1), then reduceByKey.</details>

In [ ]:
sentences = sc.parallelize(["spark is fast", "spark is distributed", "spark processes data fast"])

# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
sentences = sc.parallelize(["spark is fast", "spark is distributed", "spark processes data fast"])
word_counts = sentences.flatMap(lambda s: s.split(" ")) \
    .map(lambda w: (w, 1)) \
    .reduceByKey(lambda a, b: a + b)
print(word_counts.collect())